# 40 — Time Trends report (Proposal 1), 2012–2025

Drives `lib/trends.py` (data, cleaning, measures) and `lib/trends_charts.py` (one function per chart).
Edit chart styling in `lib/trends_charts.py`; autoreload picks up changes when you re-run a chart cell.

Data sources and caveats: `docs/data-sources.md`. Full rebuild without the notebook: `python scripts/trends_build.py`.
Re-pull external data (needs network): `python scripts/trends_pull.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.simplefilter("ignore")
import pandas as pd
from IPython.display import Image, display
from lib import trends as T, trends_charts as C
pd.set_option("display.width", 200, "display.max_columns", 40)

## Optional: refresh external data

In [ ]:
#import subprocess, sys
#subprocess.run([sys.executable, "scripts/trends_pull.py", "acs"], check=True)   # or: cpi zillow redfin fhfa

## Build the panel

In [ ]:
p = T.panel()                      # T.panel(include_ut=True) to add unorganized territories (2017+ only)
p[p.County == "Maine"].set_index("Year").loc[2012:, [
    "commitment", "commitment_real", "mill_rate", "etr", "avg_bill", "homestead_bill",
    "tax_share_income", "zhvi", "redfin_med_price", "fhfa_hpi_2000", "med_home_value", "med_owner_income", "cpi"]]

## Checks
Parcel-count repairs (Line 42) and coverage by source.

In [ ]:
_, parcel_log = T.clean_parcels(T.mvr_muni())
display(parcel_log[parcel_log.reason == "outlier"].sort_values("Year"))
cov = {v: p[p[v].notna() & (p.County != "Maine")].groupby("Year").County.nunique()
       for v in ["etr", "med_home_value", "zhvi", "redfin_med_price", "fhfa_hpi_2000"]}
pd.DataFrame(cov).loc[2012:]

## State charts
Each function saves to `output/trends/charts/state/` and returns the path.

In [ ]:
display(Image(C.property_values(p, 'Maine')))

In [ ]:
display(Image(C.property_value_index(p, 'Maine')))

In [ ]:
display(Image(C.commitment(p, 'Maine')))

In [ ]:
display(Image(C.commitment(p, 'Maine', h=4.3, name='02_commitment_standard')))   # same chart at the standard 7.6 x 4.3 in size

In [ ]:
display(Image(C.commitment_vs_cpi(p, 'Maine')))

In [ ]:
display(Image(C.commitment_vs_cpi(p, 'Maine', h=4.3, name='03_commitment_vs_cpi_standard')))   # same chart at the standard 7.6 x 4.3 in size

In [ ]:
display(Image(C.appropriations(p, 'Maine')))   # TRCF appropriations, revenues and net to be raised (2022+)

In [ ]:
display(Image(C.tax_rates(p, 'Maine')))

In [ ]:
display(Image(C.tax_bills(p, 'Maine')))

In [ ]:
display(Image(C.share_of_income(p, 'Maine')))

## One county
Any chart function takes a county name.

In [ ]:
geo = "Cumberland"
for f in C.ALL_CHARTS:
    display(Image(f(p, geo)))

## County small multiples

In [ ]:
for path in C.county_grids(p):
    display(Image(path))

## Six counties with complete data
The six counties of 65,000+ (Androscoggin, Aroostook, Cumberland, Kennebec, Penobscot, York) have ACS 1-year estimates, so every panel has every series. Saved as `output/trends/charts/county/*__acs6_counties.png`.

In [ ]:
for path in C.county_grids(p, C.ACS1_COUNTIES):
    display(Image(path))

## Growth by type of municipality
Growth in the property tax commitment (2012–2019 and 2019–2025), then in municipal appropriations (2022–2025),
by size, service-center status and economic type. Saved to `output/trends/charts/types/`.

In [ ]:
from lib import trends_types as TT
types = TT.summary()
for dim in TT.DIMS:                                   # size, service, economy
    display(Image(C.commitment_growth_by_type(types, dim)))

In [ ]:
muni = TT.trcf_summary("trcf_municipal")
for dim in TT.DIMS:
    display(Image(C.municipal_appropriation_growth_by_type(muni, dim)))
tm = TT.trcf_typed("trcf_municipal")                 # one row per town
display(Image(C.municipal_appropriation_change_distribution(tm)))
display(Image(C.municipal_appropriation_cagr_by_munitype(tm)))                 # average across towns
display(Image(C.municipal_appropriation_cagr_by_munitype(tm, stat="median")))  # median town

## Summary table for the report

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location("trends_build", "scripts/trends_build.py")
tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
summary = tb.summary(p)
summary.to_csv(T.TRENDS_OUT / "summary_by_geo.csv", index=False)
(summary.set_index("County").filter(like="cagr") * 100).round(1)   # % per year